In [7]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# --- CONFIGURATION ---
INPUT_FILE = './filtered_envisoft_air_quality_weather_data.csv' # Make sure this matches your file name
BASE_OUTPUT_DIR = 'output_correlation_station'

# --- HELPER FUNCTION: CLASSIFY REGION ---
def get_region_folder(station_name):
    """
    Determines the folder name based on the station's location keywords.
    """
    name_lower = str(station_name).lower()
    
    # 1. Hanoi & North
    if any(kw in name_lower for kw in ['hà nội']):
        return 'Hanoi_Region'
    
    # 2. HCM & South
    elif any(kw in name_lower for kw in ['hcm', 'hồ chí minh']):
        return 'HCM_Region'
    
    # 3. Danang & Central
    elif any(kw in name_lower for kw in ['đà nẵng', 'huế', 'quảng bình', 'quảng nam', 'nghệ an', 'hà tĩnh', 'thanh hóa']):
        return 'Danang_Central_Region'
    
    # 4. Fallback
    return 'Other_Stations'

# --- MAIN SCRIPT ---
# 1. Load Data
print("📥 Loading data...")
try:
    df = pd.read_csv(INPUT_FILE, dayfirst=True)
except FileNotFoundError:
    print(f"❌ Error: File '{INPUT_FILE}' not found. Please check the path.")
    exit()

df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')

columns_to_use = ['AQI', 'PM2.5', 'PM10', 'CO', 'NO2', 'O3', 'SO2',
                  'Temperature', 'Humidity', 'Pressure', 'Wind Speed']

# Convert columns to numeric, forcing errors to NaN
for col in columns_to_use:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        print(f"⚠️ Warning: Column '{col}' not found in dataset.")

# 2. Iterate Through Each Station
stations = df['Name'].unique()
print(f"👉 Found {len(stations)} stations. Processing...\n")

for station in stations:
    # Filter data for this specific station
    station_df = df[df['Name'] == station]

    # Calculate Pearson Correlation
    # min_periods=10 ensures we don't calculate correlation if there are too few data points
    corr = station_df[columns_to_use].corr(method='pearson', min_periods=10)

    # Check if the result is empty (all NaNs)
    if corr.dropna(how='all').empty:
        print(f"⚠️ Skipping '{station}': Not enough valid data points.")
        continue

    # --- DETERMINE FOLDER ---
    region_folder = get_region_folder(station)
    
    # Create the specific folder for this region
    save_dir = os.path.join(BASE_OUTPUT_DIR, region_folder)
    os.makedirs(save_dir, exist_ok=True)

    # Clean the station name for filename safety
    safe_name = str(station).replace(' ', '_').replace(':', '').replace('/', '-').replace('.', '')

    # 3. Save CSV Matrix
    csv_path = os.path.join(save_dir, f'correlation_matrix_{safe_name}.csv')
    corr.to_csv(csv_path)

    # 4. Generate & Save Heatmap
    plt.figure(figsize=(11, 9))
    
    # Using vmin=-1 and vmax=1 keeps the colors consistent (Blue=-1, Red=+1)
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", 
                linewidths=0.5, vmin=-1, vmax=1)
    
    plt.title(f'Correlation Matrix: {station}', fontsize=12, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    img_path = os.path.join(save_dir, f'heatmap_{safe_name}.png')
    plt.savefig(img_path, dpi=100)
    plt.close() # Close the plot to save memory (don't use plt.show() in a loop)

    print(f"✅ Saved: {region_folder} / {station}")

print("-" * 50)
print(f"🎉 Processing Complete! Check the '{BASE_OUTPUT_DIR}' folder.")

📥 Loading data...
👉 Found 13 stations. Processing...

✅ Saved: Hanoi_Region / Hà Nội: 556 Nguyễn Văn Cừ (KK)
✅ Saved: HCM_Region / HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận 2 (Ngã ba Lê Hữu Kiểu và Trương Văn Bang) (KK)
✅ Saved: Other_Stations / Long An: UBND Tp Tân An - 76 Hùng Vương - P.2 (KK)
✅ Saved: Hanoi_Region / Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK)
✅ Saved: Hanoi_Region / Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến (KK)
✅ Saved: Danang_Central_Region / Đà Nẵng: Khuôn viên trường ĐH sư phạm Đà Nẵng (KK)
✅ Saved: Other_Stations / Thái nguyên: Đường Hùng Vương - Tp Thái Nguyên (KK)
✅ Saved: Other_Stations / Phú Thọ: đường Hùng Vương - Tp Việt Trì (KK)
✅ Saved: Other_Stations / Bắc Giang: Khu liên cơ quan tỉnh Bắc Giang - P. Ngô Quyền - TP. Bắc Giang (KK)
✅ Saved: Other_Stations / Hà Nam: Công Viên Nam Cao - P.Quang Trung - TP. Phủ Lý (KK)
✅ Saved: Other_Stations / Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp Thành (KK)
✅ Saved: Danang_Central_Region / Quảng Bình: